In [1]:
import pandas as pd

# Load your dataset (assuming it is a CSV, adjust if using .parquet)
df = pd.read_parquet('final_data.parquet')

# List of columns that contribute to total electrical usage
usage_columns = [
    'elec_ceiling_fan_kwh',
    'elec_clothes_washer_kwh',
    'elec_cooling_kwh',
    'elec_freezer_kwh',
    'elec_heating_kwh',
    'elec_hot_water_kwh',
    'elec_lighting_exterior_kwh',
    'elec_lighting_interior_kwh',
    'elec_plug_loads_kwh',
    'elec_range_oven_kwh',
    'elec_refrigerator_kwh',
    'elec_television_kwh'
]

# Add the 'total_usage_kwh' column by summing the rows
df['total_usage_kwh'] = df[usage_columns].sum(axis=1)

# Display the first few rows to verify
# print(df[['building_id', 'total_usage_kwh']].head())

# Save the updated dataset if needed
# df.to_csv('updated_dataset.csv', index=False)

In [2]:
# ============================================================
# FHMM Appliance Disaggregation with NILMTK
# For Solar Energy Optimization Project
# ============================================================

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# --- NILMTK Imports ---
from nilmtk import DataSet, MeterGroup
from nilmtk.disaggregate import FHMM
from nilmtk.utils import print_dict
from nilmtk.metrics import f1_score, mean_absolute_error
import nilmtk.utils as utils


# ============================================================
# STEP 1 — Load & Prepare Data
# ============================================================
# NILMTK expects data in HDF5 format. If you have raw CSVs
# (e.g. from a UPS/smart meter), use the converter below first.

def load_dataset(h5_path: str, building: int = 1):
    """Load an NILMTK HDF5 dataset."""
    dataset = DataSet(h5_path)
    dataset.set_window(start="2024-01-01", end="2024-12-31")  # adjust as needed
    building_obj = dataset.buildings[building]
    elec = building_obj.elec
    return dataset, elec


# ============================================================
# STEP 2 — Convert Raw CSV to NILMTK HDF5 (if needed)
# ============================================================
# If your UPS only gives aggregate power (Watts) at intervals,
# use this helper to build a minimal NILMTK-compatible dataset.

def build_hdf5_from_df(
    aggregate_df: pd.DataFrame,    # index: datetime, columns: ['power']
    appliance_dfs: dict,           # {'fridge': df_fridge, ...}
    output_h5: str = "solar_dataset.h5"
):
    """
    Builds an HDF5 file from already loaded DataFrames.
    """
    import nilmtk.measurement as nm
    import h5py, os

    # Open HDF5 Store
    store = pd.HDFStore(output_h5, 'w', complevel=9, complib='blosc')

    def prepare_df(df):
        """Standardizes index to UTC and columns to NILMTK MultiIndex format."""
        # Ensure index is datetime and localized to UTC
        if not isinstance(df.index, pd.DatetimeIndex):
            df.index = pd.to_datetime(df.index)
        
        if df.index.tz is None:
            df.index = df.index.tz_localize('UTC')
        else:
            df.index = df.index.tz_convert('UTC')

        # Create NILMTK standard MultiIndex columns
        df.columns = pd.MultiIndex.from_tuples(
            [('power', 'active')], names=['physical_quantity', 'type']
        )
        return df

    # --- Process Aggregate (mains) ---
    agg_df = prepare_df(aggregate_df.copy())
    store.put('/building1/elec/meter1', agg_df, format='table')

    # --- Process Sub-meters (appliances) ---
    appliance_metadata = {}
    for i, (name, df) in enumerate(appliance_dfs.items(), start=2):
        sub_df = prepare_df(df.copy())
        store.put(f'/building1/elec/meter{i}', sub_df, format='table')
        appliance_metadata[i] = {'original_name': name, 'meters': [i]}

    store.close()

    # Write NILMTK metadata
    _write_metadata(output_h5, appliance_metadata)
    print(f"Dataset successfully saved to {output_h5}")
    return output_h5


def _write_metadata(h5_path: str, appliance_metadata: dict):
    """Write the minimal YAML metadata NILMTK needs."""
    import yaml, os
    base = os.path.splitext(h5_path)[0]

    building_meta = {
        'instance': 1,
        'dataset': 'SolarProject',
        'elec_meters': {
            1: {'device_model': 'UPS_aggregate', 'submeter_of': 0}
        },
        'appliances': []
    }
    for meter_i, info in appliance_metadata.items():
        building_meta['elec_meters'][meter_i] = {
            'device_model': info['original_name'],
            'submeter_of': 1
        }
        building_meta['appliances'].append({
            'original_name': info['original_name'],
            'type': info['original_name'],
            'instance': 1,
            'meters': info['meters']
        })

    dataset_meta = {
        'name': 'SolarProject',
        'long_name': 'Solar Energy Optimization Dataset',
        'buildings': {1: {'original_name': 'House 1'}}
    }

    with open(f'{base}_building1.yaml', 'w') as f:
        yaml.dump(building_meta, f)
    with open(f'{base}_dataset.yaml', 'w') as f:
        yaml.dump(dataset_meta, f)


# ============================================================
# STEP 3 — Train FHMM
# ============================================================

def train_fhmm(elec, n_states: int = 2, sample_period: int = 60):
    """
    Train an FHMM disaggregator.

    n_states      : hidden states per appliance (2 = ON/OFF, 3 = ON/STANDBY/OFF)
    sample_period : resampling period in seconds (60s = 1-min resolution)
    """
    fhmm = FHMM()

    # Select training meters — exclude the mains (meter1)
    train_elec = elec.submeters()

    print("Training FHMM on appliances:", [str(m) for m in train_elec.meters])

    fhmm.train(
        train_elec,
        sample_period=sample_period,
        # FHMM-specific params:
        num_of_states=n_states,
    )

    print("Training complete.")
    return fhmm


# ============================================================
# STEP 4 — Disaggregate Aggregate Meter
# ============================================================

def disaggregate(fhmm, mains_meter, output_h5: str = "fhmm_predictions.h5",
                 sample_period: int = 60):
    """
    Run disaggregation on the aggregate (UPS) meter.
    Returns a dict of {appliance_name: pd.Series of power (W)}
    """
    pred_store = pd.HDFStore(output_h5, 'w')

    fhmm.disaggregate(
        mains_meter.power_series_all_data(sample_period=sample_period),
        pred_store,
        sample_period=sample_period
    )
    pred_store.close()

    # Load predictions back as a dict of Series
    pred_store = pd.HDFStore(output_h5, 'r')
    predictions = {}
    for key in pred_store.keys():
        appliance_label = key.strip('/')
        predictions[appliance_label] = pred_store[key]
    pred_store.close()

    return predictions


# ============================================================
# STEP 5 — Evaluate (if ground-truth sub-meters exist)
# ============================================================

def evaluate_fhmm(predictions: dict, ground_truth_elec):
    """Compare FHMM predictions to actual sub-meter readings."""
    results = {}
    for meter in ground_truth_elec.meters:
        name = str(meter.label())
        if name not in predictions:
            continue
        gt = next(meter.power_series(sample_period=60))
        pred = predictions[name].reindex(gt.index, method='nearest')

        mae = mean_absolute_error(gt, pred)
        results[name] = {'MAE (W)': round(mae, 2)}

    return pd.DataFrame(results).T


# ============================================================
# STEP 6 — Extract Features for Downstream Scheduling Model
# ============================================================

def extract_usage_features(predictions: dict, freq: str = '1H') -> pd.DataFrame:
    """
    Aggregate per-appliance disaggregated power into hourly energy (Wh).
    This DataFrame feeds your XGBoost / RL-PPO scheduling model.

    Output columns: [timestamp, appliance_1_Wh, appliance_2_Wh, ...]
    """
    frames = {}
    for name, series in predictions.items():
        # Convert W → Wh over each hour
        hourly = series.resample(freq).mean()   # average W during hour
        frames[name] = hourly

    df = pd.DataFrame(frames)
    df.index.name = 'timestamp'

    # Add time-of-day features useful for scheduling
    df['hour'] = df.index.hour
    df['day_of_week'] = df.index.dayofweek
    df['is_weekend'] = df['day_of_week'].isin([5, 6]).astype(int)

    return df


# ============================================================
# STEP 7 — Full Pipeline
# ============================================================

def run_fhmm_pipeline(
    h5_path: str,
    building: int = 1,
    n_states: int = 2,
    sample_period: int = 60,
    pred_output: str = "fhmm_predictions.h5",
    features_output: str = "usage_features.csv"
):
    print("=== FHMM Disaggregation Pipeline ===\n")

    # 1. Load
    dataset, elec = load_dataset(h5_path, building)
    mains = elec.mains()
    print(f"Mains meter: {mains}")
    print(f"Appliance meters: {[str(m) for m in elec.submeters().meters]}\n")

    # 2. Train
    fhmm = train_fhmm(elec, n_states=n_states, sample_period=sample_period)

    # 3. Disaggregate
    print("\nDisaggregating aggregate meter...")
    predictions = disaggregate(fhmm, mains, pred_output, sample_period)
    print(f"Disaggregated {len(predictions)} appliances.")

    # 4. Evaluate (if submeters available)
    try:
        eval_df = evaluate_fhmm(predictions, elec.submeters())
        print("\nEvaluation Results:")
        print(eval_df.to_string())
    except Exception as e:
        print(f"(Skipping evaluation: {e})")

    # 5. Extract features for scheduling model
    features_df = extract_usage_features(predictions)
    features_df.to_csv(features_output)
    print(f"\nUsage features saved to '{features_output}'")
    print(features_df.head())

    return fhmm, predictions, features_df


# ============================================================
# Entry Point
# ============================================================

if __name__ == "__main__":
    # --- Option A: You already have an HDF5 (e.g. REDD, UK-DALE) ---
    # fhmm, preds, features = run_fhmm_pipeline("redd.h5")

    # --- Option B: Build HDF5 from your own CSVs first ---
    h5 = build_hdf5_from_df(
        aggregate_df = df,       # timestamp, power
        appliance_csvs={                          # only needed for training
            "fridge":     "fridge.csv",
            "ac":         "ac.csv",
            "washing_machine": "washing_machine.csv",
            "tv":         "tv.csv",
        },
        output_h5="solar_dataset.h5"
    )
    fhmm, preds, features = run_fhmm_pipeline(h5)

ModuleNotFoundError: No module named 'nilmtk'